In [1]:
import arcpy

# --- USER INPUTS ----------------------------------------------------
input_layer = "OrganicsData_V1"   # layer name in your map TOC
output_gdb = r"Z:\users\anna_abramova\Derrick_ISRAD_Map\Derrick_ISRAD_Map.gdb"   # <- change to your .gdb path
output_name = "OrganicsData_V1_attr_points"
# --------------------------------------------------------------------

# Attribute fields containing 0/1
attr_fields = [
    'lyr_14C_01',
    'int_14C_01',
    'flux_14C_01',
    'frac_14C_01',
    'incub_14C_01'   # <--- new field added here
]

# Describe input to get spatial reference & extent
desc = arcpy.Describe(input_layer)
sp_ref = desc.spatialReference
ext = desc.extent

# Create a small offset distance based on layer extent (in map units)
extent_min = min(ext.XMax - ext.XMin, ext.YMax - ext.YMin)
offset_dist = extent_min * 0.005   # you can change 0.005 if needed

print(f"Using offset distance: {offset_dist}")

# Set fixed offset positions for each attribute
# Now using 5-point "star" pattern: N, E, S, W, NE
offsets_by_attr = {
    'lyr_14C_01':   (0, offset_dist),             # up
    'int_14C_01':   (offset_dist, 0),             # right
    'flux_14C_01':  (0, -offset_dist),            # down
    'frac_14C_01':  (-offset_dist, 0),            # left
    'incub_14C_01': (offset_dist * 0.7, offset_dist * 0.7)  # diagonal NE
}

# Create output feature class
arcpy.env.workspace = output_gdb

if arcpy.Exists(output_name):
    arcpy.Delete_management(output_name)

out_fc = arcpy.management.CreateFeatureclass(
    out_path=output_gdb,
    out_name=output_name,
    geometry_type="POINT",
    spatial_reference=sp_ref
)[0]

# Add fields: original OID and attribute name
arcpy.management.AddField(out_fc, "orig_oid", "LONG")
arcpy.management.AddField(out_fc, "attr_name", "TEXT", field_length=50)

# Build cursor fields list
cursor_fields = ["OID@", "SHAPE@XY"] + attr_fields

# Create points
with arcpy.da.SearchCursor(input_layer, cursor_fields) as s_cur, \
     arcpy.da.InsertCursor(out_fc, ["orig_oid", "attr_name", "SHAPE@XY"]) as i_cur:

    for row in s_cur:
        oid = row[0]
        x, y = row[1]
        attr_values = dict(zip(attr_fields, row[2:]))

        for attr_name, val in attr_values.items():
            # Support values that may be string or numeric
            try:
                is_present = int(val) == 1
            except (TypeError, ValueError):
                is_present = False

            if is_present:
                dx, dy = offsets_by_attr[attr_name]
                new_xy = (x + dx, y + dy)
                i_cur.insertRow((oid, attr_name, new_xy))

print(f"Created: {out_fc}")

# Optionally add new layer to map
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.listMaps()[2]
m.addDataFromPath(out_fc)
print("Added output layer to the map.")


Using offset distance: 0.4058500000000001
Created: Z:\users\anna_abramova\Derrick_ISRAD_Map\Derrick_ISRAD_Map.gdb\OrganicsData_V1_attr_points
Added output layer to the map.
